In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [4]:
%pip install langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [6]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='5cb761af-311e-4cf6-96ee-5b1d52dd279e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 271, 'total_tokens': 363, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Ef35Im34k3ZNhloZW79GR88bhbFqq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a19211-ecbb-7fa0-9892-d2ea63e3181d-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters'}, 'id': 'call_HRkItlez96dzmN1rOhzga92I', 'type': 'tool_call'}], invalid_tool_calls=[], usag

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "mcpServers": {
            "time": {
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
            }
        }
    }
)

tools = await client.get_tools()

> Configurar el cliente para que ejecute el comando usando el entorno de comandos de Windows (cmd.exe). Modifica tu configuración de la siguiente manera:

In [33]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "time": {
            # Se fuerza el uso de cmd /c para que Windows reconozca el comando 'uvx'
            "command": "cmd.exe",
            "args": [
                "/c",
                "uvx",
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ],
            "transport": "stdio"
        }
    }
)

tools = await client.get_tools()

In [34]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [35]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='fd606104-5943-4b98-b744-dbd030f6c52f'),
              AIMessage(content='I don’t have your local clock. Which time zone should I use—UTC or your city/time zone? If you tell me your location (or time zone), I’ll give you the current time there and can also help with conversions.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 825, 'prompt_tokens': 129, 'total_tokens': 954, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Ef3ZXREh7slkcDC77LLQrokYhTaCB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1922e-8b35-7fe3-ab11-48c764b110d4-0', 